# GSPO

$$
\mathcal{J}_{\text{GSPO}}(\theta) = \mathbb{E}_{q \sim P(Q), \{o_i\}_{i=1}^G \sim \pi_{\theta_{\text{old}}}(O \mid q)} \Bigg[ \frac{1}{G} \sum_{i=1}^G \min \Bigg( s_i(\theta) \hat{A}_i, \text{clip} \left( s_i(\theta), 1 - \epsilon_{\text{low}}, 1 + \epsilon_{\text{high}} \right) \hat{A}_i \Bigg) \Bigg]
$$

$$
s_i(\theta)=\exp \left( \frac{1}{|o_i|} \sum_{t=1}^{|o_i|} \log \frac{\pi_\theta(o_{i,t} \mid q, o_{i,<t})}{\pi_{\theta_{\text{old}}}(o_{i,t} \mid q, o_{i,<t})} \right), \quad \hat{A}_i=\frac{r_i-\text{mean}(\mathbf{r})}{\text{std}(\mathbf{r})}
$$

In [ ]:
import torch

def gspo_loss(rewards, logp_per_token, old_logp_per_token, completion_mask=None, clip_epsilon_low=3e-4, clip_epsilon_high=4e-4):
    """
    Args:
        rewards (torch.Tensor): 奖励, shape: [batch_size, num_generation]
        logp_per_token (torch.Tensor): 策略模型logp, shape: [batch_size, num_generation, seq_len]
        old_logp_per_token (torch.Tensor): 旧策略模型logp, shape: [batch_size, num_generation, seq_len]
        completion_mask (torch.Tensor | None): 有效token掩码, shape: [batch_size, num_generation, seq_len]
        clip_epsilon_low (float): 下界裁剪参数
        clip_epsilon_high (float): 上界裁剪参数
    """
    if completion_mask is None:
        completion_mask = torch.ones_like(logp_per_token)

    mean_grouped_rewards = rewards.mean(dim=-1, keepdim=True) # shape: [batch_size, 1]
    std_grouped_rewards = rewards.std(dim=-1, keepdim=True)   # shape: [batch_size, 1]
    advantage = (rewards - mean_grouped_rewards) / (std_grouped_rewards + 1e-8) # [batch_size, num_generation]

    seq_len = completion_mask.sum(dim=-1).clamp_min(1.0) # [batch_size, num_generation]
    sequence_log_ratio = ((logp_per_token - old_logp_per_token) * completion_mask).sum(dim=-1) / seq_len
    sequence_ratio = torch.exp(sequence_log_ratio) # [batch_size, num_generation]
    clipped_sequence_ratio = torch.clamp(sequence_ratio, 1.0 - clip_epsilon_low, 1.0 + clip_epsilon_high)

    objective1 = sequence_ratio * advantage
    objective2 = clipped_sequence_ratio * advantage
    sequence_objective = torch.min(objective1, objective2) # [batch_size, num_generation]

    loss = -sequence_objective.mean()
    return loss
